# Extension 3: ML-Based Anomaly Detection

**Goal:** Evaluate whether a KMeans distance-based approach detects more or fewer anomalies
than the existing Z-score method, by running both independently and comparing results
side-by-side.

**This notebook does NOT modify `src/transforms/performance/anomalies.py`.** It reads
the Z-score detections already written to `performance_anomalies` by Job 3, then runs
KMeans against the same raw `performance_by_version` data.

**Why KMeans for anomaly detection:**
- Handles multivariate data (p50 + p95 + p99 simultaneously)
- Does not assume a Gaussian distribution
- The 99th-percentile distance threshold adapts to the actual data shape

**Data sources:**
| Table | Role |
|---|---|
| `performance_by_version` | Raw performance metrics to cluster |
| `device_performance` | Secondary validation dataset |
| `performance_anomalies` | Existing Z-score detections for comparison |

---
**Prerequisites:** Run `make run-jobs` before opening this notebook.

## Cell 1 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    with psycopg2.connect(**PG) as conn:
        return pd.read_sql(sql, conn)

spark = (
    SparkSession.builder
    .appName("ML Feasibility — Anomaly Detection")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Load Performance Data

In [ ]:
perf_pd = pg_query("""
    SELECT app_version, metric_date,
           p50_duration_ms, p95_duration_ms, p99_duration_ms,
           total_interactions
    FROM performance_by_version
    ORDER BY metric_date
""")

print(f"Rows loaded: {len(perf_pd):,}")
print(f"App versions: {perf_pd['app_version'].nunique()}")
print(f"Date range  : {perf_pd['metric_date'].min()} → {perf_pd['metric_date'].max()}")
perf_pd.head()

## Cell 3 — KMeans Pipeline

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql import functions as F

sdf = spark.createDataFrame(perf_pd.fillna(0))

PERF_COLS = ["p50_duration_ms", "p95_duration_ms", "p99_duration_ms", "total_interactions"]

assembler = VectorAssembler(inputCols=PERF_COLS, outputCol="features_raw")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True)
kmeans    = KMeans(featuresCol="features", k=4, seed=42)
pipeline  = Pipeline(stages=[assembler, scaler, kmeans])

print("Fitting KMeans pipeline (k=4)...")
model = pipeline.fit(sdf)
print("Done.")

# Print cluster sizes
preds_tmp = model.transform(sdf)
print("\nCluster sizes:")
preds_tmp.groupBy("prediction").count().orderBy("prediction").show()

## Cell 4 — Centroid Distance Scoring

For each row, compute the Euclidean distance from its scaled feature vector to its assigned
cluster centroid. Rows whose distance exceeds the 99th percentile are flagged as anomalies.

In [ ]:
centers = model.stages[-1].clusterCenters()  # list of numpy arrays

@F.udf("double")
def centroid_dist(features, cluster_idx):
    return float(np.linalg.norm(np.array(features.toArray()) - centers[cluster_idx]))

predictions = model.transform(sdf).withColumn(
    "centroid_distance",
    centroid_dist(F.col("features"), F.col("prediction"))
)

# Flag as anomaly if distance exceeds the 99th percentile
p99_dist = predictions.approxQuantile("centroid_distance", [0.99], 0.01)[0]
print(f"99th-percentile centroid distance: {p99_dist:.4f}")

ml_anomalies = predictions.filter(F.col("centroid_distance") > p99_dist)
print(f"ML anomalies detected: {ml_anomalies.count()}")

## Cell 5 — KMeans Anomaly Details

In [ ]:
ml_pd = (
    ml_anomalies
    .select("metric_date", "app_version", "centroid_distance",
            "p50_duration_ms", "p95_duration_ms", "p99_duration_ms")
    .orderBy(F.col("centroid_distance").desc())
    .toPandas()
)
print(f"KMeans anomalies (top rows):")
ml_pd.head(10)

## Cell 6 — Load Existing Z-Score Anomalies

In [ ]:
zscore_pd = pg_query("""
    SELECT metric_date, app_version, severity, z_score
    FROM performance_anomalies
    WHERE app_version IS NOT NULL
    ORDER BY metric_date
""")

print(f"Z-score anomalies in DB: {len(zscore_pd)}")
if len(zscore_pd) > 0:
    print("Severity distribution:")
    print(zscore_pd["severity"].value_counts().to_string())
zscore_pd.head()

## Cell 7 — Side-by-Side Comparison

In [ ]:
print(f"Z-score anomalies : {len(zscore_pd)}")
print(f"KMeans anomalies  : {len(ml_pd)}")

if len(zscore_pd) > 0 and len(ml_pd) > 0:
    # Normalise dates to string for merge
    zscore_pd["metric_date"] = zscore_pd["metric_date"].astype(str)
    ml_pd["metric_date"]     = ml_pd["metric_date"].astype(str)

    overlap = pd.merge(
        zscore_pd[["metric_date", "app_version"]],
        ml_pd[["metric_date", "app_version"]],
        on=["metric_date", "app_version"],
    )
    print(f"Detected by both  : {len(overlap)}")
    print(f"Z-score only      : {len(zscore_pd) - len(overlap)}")
    print(f"KMeans only       : {len(ml_pd) - len(overlap)}")
else:
    print("One or both anomaly sets are empty — cannot compute overlap.")

## Cell 8 — Centroid Distance Distribution

In [ ]:
import matplotlib.pyplot as plt

dist_pd = predictions.select("centroid_distance").toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dist_pd["centroid_distance"], bins=40, edgecolor="white", color="steelblue")
ax.axvline(p99_dist, color="red", linestyle="--", label=f"p99 threshold ({p99_dist:.2f})")
ax.set_xlabel("Centroid Distance (scaled space)")
ax.set_ylabel("Count")
ax.set_title("KMeans Centroid Distance Distribution — performance_by_version")
ax.legend()
plt.tight_layout()
plt.show()

## Cell 9 — Device Performance Validation

Run the same KMeans approach on `device_performance` as a cross-validation check.

In [ ]:
device_pd = pg_query("""
    SELECT device_type, metric_date,
           p50_duration_ms, p95_duration_ms, p99_duration_ms
    FROM device_performance
    ORDER BY metric_date
""")

DEVICE_COLS = ["p50_duration_ms", "p95_duration_ms", "p99_duration_ms"]
sdf_dev = spark.createDataFrame(device_pd.fillna(0))

asm_dev  = VectorAssembler(inputCols=DEVICE_COLS, outputCol="features_raw")
scl_dev  = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True)
km_dev   = KMeans(featuresCol="features", k=3, seed=42)
pipe_dev = Pipeline(stages=[asm_dev, scl_dev, km_dev])

model_dev  = pipe_dev.fit(sdf_dev)
centers_dev = model_dev.stages[-1].clusterCenters()

preds_dev = model_dev.transform(sdf_dev).withColumn(
    "centroid_distance",
    centroid_dist(F.col("features"), F.col("prediction"))
)

p99_dev = preds_dev.approxQuantile("centroid_distance", [0.99], 0.01)[0]
dev_anomalies = preds_dev.filter(F.col("centroid_distance") > p99_dev)
print(f"Device performance ML anomalies: {dev_anomalies.count()}")
dev_anomalies.select("device_type", "metric_date", "centroid_distance").orderBy(
    F.col("centroid_distance").desc()
).show(10)

## Cell 10 — Cleanup

In [ ]:
spark.stop()
print("Spark session stopped.")